In [2]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import pandas as pd
import muon as mu

np.random.seed(42)
import random
random.seed(42)

In [3]:
mdata = mu.read("./data/EAE/CCA/merged_EAE_CD4_cloned_singlePerClone.h5mu")
mdata

MuData object with n_obs × n_vars = 4840 × 4000
  obs:	'GSE', 'GSM', 'VDJ_1_cdr3_aa', 'VDJ_1_j_call', 'VDJ_1_v_call', 'VJ_1_cdr3_aa', 'VJ_1_j_call', 'VJ_1_v_call', 'cell_type', 'condition', 'sample_id', 'state', 'VDJ_1_cdr3_aa_length', 'VJ_1_cdr3_aa_length', 'set'
  uns:	'cca_x_weights', 'cca_y_weights', 'lr_coef', 'lr_intercept'
  obsm:	'VDJ_1_j_call', 'VDJ_1_v_call', 'VJ_1_j_call', 'VJ_1_v_call', 'X_VDJ_1_cdr3_aa_atchley', 'X_VDJ_1_cdr3_aa_atchley_pairwise', 'X_VDJ_1_cdr3_aa_composition', 'X_VJ_1_cdr3_aa_atchley', 'X_VJ_1_cdr3_aa_atchley_pairwise', 'X_VJ_1_cdr3_aa_composition', 'tcr_embs'
  2 modalities
    gex:	4840 x 4000
      obs:	'sample_id', 'date', 'tissue', 'sample', 'mouse_BC', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'n_genes', 'n_counts', 'CD4score', 'CD8score', 'Tregscore', 'Th17score', 'cell_type', 'IFN_stimscore', 'Activationscore', 'Exhaustscore', 'Mem_Naivescore', 'state', 'TCR_clonotype_frequency', 'condition', 'batch', 'leiden', 'GSE', 'Tissue_group', 'CV_score_0', 'CV_score_1', 'CV_score_2', 'CV_score_3', 'CV_score_0_high', 'CV_score_1_high', 'CV_score_2_high', 'CV_score_3_high'
      var:	'gene_ids', 'feature_types', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
      uns:	'GSE_colors', 'X_umap_harmony', 'cell_type_colors', 'hvg', 'leiden', 'neighbors', 'neighbors_harmony', 'pca', 'state_colors', 'tissue_colors', 'umap'
      obsm:	'X_pca', 'X_pca_harmony', 'X_umap', 'X_umap_harmony'
      varm:	'PCs'
      obsp:	'connectivities', 'distances', 'neighbors_harmony_connectivities', 'neighbors_harmony_distances'
    airr:	4840 x 0
      obs:	'sample_id', 'receptor_type', 'receptor_subtype', 'chain_pairing', 'clone_id', 'clone_id_size', 'clonal_expansion', 'sample', 'tissue', 'condition', 'GSE'
      obsm:	'airr', 'chain_indices'

In [15]:
tcr_embs = pd.DataFrame(mdata.obsm['tcr_embs'], index=mdata.obs_names)
tcr_embs.head(5)

,0,1,2,3,4,5,6,7,8,9,...,1307,1308,1309,1310,1311,1312,1313,1314,1315,1316
CAGCATATCCCGTAAA-1_0516_CNS,0.0,0.0,0.29947,-0.329505,0.000863,0.306644,-0.093899,0.236783,0.378501,-0.413445,...,-0.05937,-0.02876,-0.072056,-0.024904,-0.081582,-0.057591,-0.123748,0.0,-0.321942,1.079007
ATGAGGAGTCTATTCG-1_0516_CNS,0.0,0.0,0.29947,-0.329505,0.000863,0.306644,-0.093899,0.236783,0.378501,-0.413445,...,-0.05937,-0.02876,-0.072056,-0.024904,-0.081582,-0.057591,-0.123748,0.0,0.653761,0.059804
GCATAAGCACTCGACA-1_0516_CNS,0.0,0.0,0.29947,-0.329505,0.000863,0.306644,-0.093899,0.236783,0.378501,-0.413445,...,-0.05937,-0.02876,-0.072056,-0.024904,-0.081582,-0.057591,-0.123748,0.0,0.653761,-0.959399
CCCAACACAGCGTATT-1_0516_CNS,0.0,0.0,0.29947,-0.329505,0.000863,0.306644,-0.093899,0.236783,0.378501,-0.413445,...,-0.05937,-0.02876,-0.072056,-0.024904,-0.081582,-0.057591,-0.123748,0.0,-0.321942,0.059804
GGGTATGAGACGGGTA-1_0516_CNS,0.0,0.0,0.29947,-0.329505,0.000863,0.306644,-0.093899,0.236783,0.378501,-0.413445,...,-0.05937,-0.02876,-0.072056,-0.024904,-0.081582,-0.057591,-0.123748,0.0,-0.321942,1.079007


In [12]:
gex_df = mdata['gex'].to_df()
gex_df = gex_df.loc[:, ~gex_df.columns.str.startswith('mt-')]
gex_df.head(5)

,1110008F13Rik,1110034G24Rik,1110037F02Rik,1500009L16Rik,1700001K23Rik,1700003C15Rik,1700012B07Rik,1700016L21Rik,1700016P03Rik,1700019D03Rik,...,Zscan18,Zswim6,Zup1,Zzef1,Zzz3,mt-Atp6,mt-Cytb,mt-Nd3,mt-Nd5,mt-Nd6
CAGCATATCCCGTAAA-1_0516_CNS,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.256863,1.256863,0.000000,0.0,2.608001,2.608001,0.0,1.796541,1.256863
ATGAGGAGTCTATTCG-1_0516_CNS,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.000000,0.000000,0.0,2.478547,2.855659,0.0,1.865933,0.000000
GCATAAGCACTCGACA-1_0516_CNS,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.291599,0.000000,0.841265,0.0,2.447066,2.898720,0.0,2.187824,0.000000
CCCAACACAGCGTATT-1_0516_CNS,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.000000,0.000000,0.0,2.575745,2.575745,0.0,1.955938,0.000000
GGGTATGAGACGGGTA-1_0516_CNS,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.000000,0.000000,0.0,2.986015,2.342123,0.0,2.342123,0.000000


In [7]:
labels = mdata['gex'].obs[['tissue', 'cell_type', 'state', 'GSE', 'sample_id']]
labels = pd.concat([labels, mdata.obs['set']], axis=1)
labels.columns = [f"label_{col}" for col in labels.columns]

labels.head(5)

,label_tissue,label_cell_type,label_state,label_GSE,label_sample_id,label_set
CAGCATATCCCGTAAA-1_0516_CNS,CNS,CD4,Activation,LEE,5_3,train
ATGAGGAGTCTATTCG-1_0516_CNS,CNS,NaN,Activation,LEE,5_3,train
GCATAAGCACTCGACA-1_0516_CNS,CNS,Treg,IFN_stim,LEE,5_3,train
CCCAACACAGCGTATT-1_0516_CNS,CNS,NaN,Activation,LEE,5_3,train
GGGTATGAGACGGGTA-1_0516_CNS,CNS,CD4,Exhaust,LEE,5_4,train


In [16]:
tcr_embs.to_csv("tcr_embs.csv", index=True)
gex_df.to_csv("gex_df.csv", index=True)
labels.to_csv("labels.csv", index=True)
